In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib
import json
import os

# --- 1. Generate Synthetic Patient Data ---
print("Generating synthetic dataset...")
np.random.seed(42)
n_samples = 5000

# Simulating health metrics
data = {
    'age': np.random.normal(65, 12, n_samples).astype(int),
    'bmi': np.random.normal(28, 5, n_samples),
    'blood_pressure': np.random.normal(130, 15, n_samples),
    'previous_admissions': np.random.poisson(1.5, n_samples),
    'cholesterol': np.random.normal(200, 40, n_samples)
}
df = pd.DataFrame(data)

# Create a target variable (Readmission) based on some underlying logic
# Higher age, higher BMI, and more previous admissions increase readmission risk
risk_score = (df['age'] * 0.02) + (df['bmi'] * 0.05) + (df['previous_admissions'] * 0.5) + np.random.normal(0, 1, n_samples)
df['readmitted'] = (risk_score > risk_score.median()).astype(int)

# --- 2. Train the ML Model ---
print("Training the baseline model...")
X = df.drop('readmitted', axis=1)
y = df['readmitted']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest usually hits that 75-85% sweet spot easily
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# --- 3. Evaluate Metrics ---
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Model Trained Successfully!")
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC:  {roc_auc:.4f}")

# --- 4. Save Model & Baseline Distributions (CRITICAL FOR MLOPS) ---
print("Saving model and baseline metrics...")

# Ensure a directory exists for our artifacts
os.makedirs("artifacts", exist_ok=True)

# Save the model
joblib.dump(model, 'artifacts/model.joblib')

# Save the baseline distribution of the training data
# We use .item() to convert NumPy data types to native Python types for JSON serialization
baseline_stats = {}
for col in X_train.columns:
    baseline_stats[col] = {
        'mean': X_train[col].mean().item(),
        'std': X_train[col].std().item(),
        'min': X_train[col].min().item(),
        'max': X_train[col].max().item()
    }

with open('artifacts/baseline_stats.json', 'w') as f:
    json.dump(baseline_stats, f, indent=4)

print("Setup complete. Artifacts saved in '/artifacts'.")

Generating synthetic dataset...
Training the baseline model...
Model Trained Successfully!
Accuracy: 0.7130
ROC-AUC:  0.7703
Saving model and baseline metrics...
Setup complete. Artifacts saved in '/artifacts'.
